<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/day11-discussion-3.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Day 11, Part 3 discussion — Train on eukaryotes, test on bacteria

SignalP 5.0 needed to be told the organism group of every sequence.
SignalP 6.0 does not: its authors report that the protein language model
infers the taxonomic context from the sequence itself.

Test that idea on our data. Train signal-peptide detectors on
**eukaryotic** proteins only, test them on **bacterial** proteins (and the
other way round), from one-hot sequences and from frozen ESM-2
embeddings. Bacterial and eukaryotic signal peptides share the
n/h/c-region plan but differ in details (bacterial ones tend to be
longer, with more positively charged n-regions).

**Predict first:** which classifier transfers across kingdoms better, and
in which direction is transfer harder? (On Colab: `!pip install fair-esm`.)

In [ ]:
import io, os, requests, numpy as np, pandas as pd, torch, esm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import matthews_corrcoef

URL = "https://raw.githubusercontent.com/ElofssonLab/kb8029-book/main/notebooks/data/day11-signal-peptides.tsv"
LOCAL = "data/day11-signal-peptides.tsv"
table = pd.read_csv(LOCAL if os.path.exists(LOCAL) else io.StringIO(requests.get(URL, timeout=60).text), sep="\t")
y = table.sp_end.notna().to_numpy().astype(int)
eukaryote = table.group.str.startswith("euk").to_numpy()
print(f"eukaryotes: {eukaryote.sum()} proteins ({y[eukaryote].sum()} with SP); "
      f"bacteria: {(~eukaryote).sum()} ({y[~eukaryote].sum()} with SP)")

N = 70
AA = "ACDEFGHIKLMNPQRSTVWY"
onehot = np.zeros((len(table), N * 20), dtype=np.float32)
for n, s in enumerate(table.sequence):
    for p, ch in enumerate(s[:N]):
        onehot[n, p * 20 + AA.index(ch)] = 1.0

model, alphabet = esm.pretrained.esm2_t12_35M_UR50D(); model.eval()
convert = alphabet.get_batch_converter()
emb = []
with torch.no_grad():
    for b in range(0, len(table), 32):
        _, _, tok = convert([(str(k), s[:N]) for k, s in enumerate(table.sequence[b:b + 32])])
        emb.append(model(tok, repr_layers=[12])["representations"][12][:, 1:N + 1].mean(1).numpy())
emb = np.concatenate(emb)
print("features:", onehot.shape, emb.shape)

In [ ]:
print(f"{'features':10s} {'train euk -> test bac':>24s} {'train bac -> test euk':>24s}")
for name, F in [("one-hot", onehot), ("ESM-2", emb)]:
    a = LogisticRegression(max_iter=3000).fit(F[eukaryote], y[eukaryote]).predict(F[~eukaryote])
    b = LogisticRegression(max_iter=3000).fit(F[~eukaryote], y[~eukaryote]).predict(F[eukaryote])
    print(f"{name:10s} {matthews_corrcoef(y[~eukaryote], a):24.3f} {matthews_corrcoef(y[eukaryote], b):24.3f}")

**Discuss:**

1. How much does each classifier lose when the test kingdom differs from
   the training kingdom? (Compare with the mixed-kingdom MCC on the book
   page.)
2. Why might transfer from bacteria to eukaryotes be harder than the
   other way round? Look at the number of training proteins in each
   direction.
3. SignalP 6.0 no longer asks for the organism group, and is advertised
   for metagenomic proteins of unknown origin. Do these numbers support
   that choice? What would you still check before trusting it on archaea,
   which are absent from our dataset?